# Last Modified: 2026-04-14 16:15:00

---

# LFP Battery SOH Preprocessing Pipeline (Optimized Batch Version)

This notebook implements a physically-consistent preprocessing framework for the entire dataset:
1. **Phase 1**: Global Physical Cleaning (Unit conversion and noise smoothing for ALL cells)
2. **Phase 2**: Scenario-based Slicing Definition
3. **Phase 2.1**: **Global Slicing Loop (Generate & Cache Raw Segment Pool)**
4. **Phase 3**: 40D HI Extraction Logic Definition
5. **Phase 4**: **Feature Extraction & Scaling (Reuse Sliced Data)**
6. **Phase 5**: Global Tensor Saving

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm.notebook import tqdm
import os
import sys
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loaders import load_hust, load_mit

: 

## Phase 0: Data Loading

In [ ]:
DATASET_TYPE = "hust"
all_cells = load_hust(PROJECT_ROOT / "HUST_data" / "data") if DATASET_TYPE == "hust" else load_mit(PROJECT_ROOT / "MIT_data")
print(f"Loaded {len(all_cells)} cells for {DATASET_TYPE.upper()} dataset.")

## Phase 1: Global Physical Cleaning & Unit Conversion

In [ ]:
cleaning_reports = []

def clean_physical_data(df, cid, cyc):
    clean_df = df.copy()
    
    # 1. Unit Conversion
    if "Current (mA)" in clean_df.columns: 
        clean_df["Current (A)"] = clean_df["Current (mA)"] / 1000.0
    
    num_cols = clean_df.select_dtypes(include=[np.number]).columns
    
    # 2. Outlier detection (Z-score >= 7)
    outlier_mask = pd.Series(False, index=clean_df.index)
    outlier_counts = {col: 0 for col in num_cols}
    
    for col in num_cols:
        col_std = clean_df[col].std()
        if col_std > 1e-6:
            z_scores = np.abs((clean_df[col] - clean_df[col].mean()) / col_std)
            is_outlier = z_scores >= 7
            if is_outlier.any():
                outlier_counts[col] = is_outlier.sum()
                outlier_mask |= is_outlier
    
    # 3. Remove outlier rows
    if outlier_mask.any():
        clean_df = clean_df[~outlier_mask].copy()
    
    # 4. Interpolation and Filling
    if not clean_df.empty:
        for col in num_cols:
            nan_count = clean_df[col].isna().sum()
            o_count = outlier_counts[col]
            if o_count > 0 or nan_count > 0:
                cleaning_reports.append({
                    "cell_id": cid,
                    "cycle": cyc,
                    "feature": col,
                    "outlier_removed": o_count,
                    "interpolated_count": nan_count
                })
        
        clean_df[num_cols] = clean_df[num_cols].interpolate(method='linear', limit_direction='both')
        clean_df = clean_df.ffill().bfill()
        
        # 5. Noise Smoothing (Savitzky-Golay)
        v_cols = [c for c in clean_df.columns if "Voltage" in c]
        if v_cols and len(clean_df) >= 11:
            v_col = v_cols[0]
            clean_df[v_col] = savgol_filter(clean_df[v_col], window_length=11, polyorder=3)
            
    return clean_df

for cid, cell in all_cells.items():
    for cyc in tqdm(list(cell.data.keys()), desc=f"Cleaning Cell {cid}"): 
        cell.data[cyc] = clean_physical_data(cell.data[cyc], cid, cyc)

# Save cleaning report
report_df = pd.DataFrame(cleaning_reports)
report_path = PROJECT_ROOT / "outputs" / "cleaning_report.csv"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_df.to_csv(report_path, index=False)
print(f"Cleaning report saved to {report_path}")

## Phase 2: Scenario-based Slicing Definition

In [ ]:
def phase2_slice_data(df, start_p=0.0, end_p=1.0, mode="C"): 
    col_i = "Current (A)" if "Current (A)" in df.columns else [c for c in df.columns if "Current" in c][0]
    mode_df = df[df[col_i] > 0.01] if mode == "C" else df[df[col_i] < -0.01]
    if mode_df.empty: return mode_df, 0, (1 if mode == "C" else 0)
    n = len(mode_df)
    s_idx, e_idx = int(n * start_p), int(n * end_p)
    sliced = mode_df.iloc[s_idx:e_idx]
    avg_pos = (start_p + end_p) / 2
    soc_label = -2 if avg_pos <= 0.3 else (-1 if avg_pos <= 0.7 else 0)
    mode_label = 1 if mode == "C" else 0
    return sliced, soc_label, mode_label

## Phase 2.1: Global Slicing Loop (Generate & Cache Raw Segment Pool)

Generate all raw data segments. Saves to a pkl file to skip processing in future runs.

In [ ]:
import os
import gc
import pickle
import numpy as np
from tqdm import tqdm
from collections import defaultdict

# 1. 경로 설정 및 분할 저장용 디렉토리 생성
CACHE_DIR = PROJECT_ROOT / "outputs" / "processed" / f"{DATASET_TYPE}_chunks"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FINAL_CACHE_PATH = PROJECT_ROOT / "outputs" / "processed" / f"{DATASET_TYPE}_raw_segments_cache.pkl"

# 배치별 캐시 로드 함수
def load_all_batches(cache_dir):
    all_raw_segments = []
    full_cycles_map = {}
    batch_files = sorted(cache_dir.glob("batch_*.pkl"))
    if not batch_files:
        return None, None
    print(f"Loading {len(batch_files)} batch files from {cache_dir}")
    for batch_file in batch_files:
        with open(batch_file, "rb") as f:
            batch_data = pickle.load(f)
        all_raw_segments.extend(batch_data['segments'])
        full_cycles_map.update(batch_data['full_cycles'])
        del batch_data
        gc.collect()
    return all_raw_segments, full_cycles_map

# 2. 배치 파일이 이미 존재하면 로드, 없으면 배치별 처리
existing_batches = sorted(CACHE_DIR.glob("batch_*.pkl"))

if existing_batches:
    print(f"Found {len(existing_batches)} existing batch files. Loading...")
    all_raw_segments, full_cycles_map = load_all_batches(CACHE_DIR)
    print(f"Successfully loaded {len(all_raw_segments)} segments.")

else:
    scen_bounds = {"H": (0.0, 0.3), "M": (0.3, 0.7), "L": (0.7, 1.0), "Random": (0.0, 1.0)}
    lengths = [0.1, 0.2, 0.3]
    step = 0.1

    # 셀을 배치 ID 기준으로 그룹핑 ("1-1" → batch_id=1)
    batch_groups = defaultdict(dict)
    for cid, cell in all_cells.items():
        batch_id = int(str(cid).split("-")[0])
        batch_groups[batch_id][cid] = cell

    all_raw_segments = []   # 최종 집계용 (배치 저장 후 비움)
    full_cycles_map = {}    # 최종 집계용 (배치 저장 후 비움)

    for batch_id in sorted(batch_groups.keys()):
        batch_cache_path = CACHE_DIR / f"batch_{batch_id}.pkl"

        # 이미 처리된 배치는 스킵
        if batch_cache_path.exists():
            print(f"[Batch {batch_id}] Already processed, skipping.")
            with open(batch_cache_path, "rb") as f:
                batch_data = pickle.load(f)
            all_raw_segments.extend(batch_data['segments'])
            full_cycles_map.update(batch_data['full_cycles'])
            del batch_data
            gc.collect()
            continue

        batch_cells = batch_groups[batch_id]
        batch_segments = []
        batch_full_cycles = {}

        print(f"\n[Batch {batch_id}] Processing {len(batch_cells)} cells: {list(batch_cells.keys())}")

        for cid, cell in batch_cells.items():
            for cyc in tqdm(sorted(list(cell.data.keys())), desc=f"  Cell {cid}"):
                df = cell.data[cyc]

                # numeric 컬럼만 float32로 변환
                num_cols = df.select_dtypes(include=[np.number]).columns
                df[num_cols] = df[num_cols].astype(np.float32)

                # Full cycle 데이터 추출
                full_c_df, _, _ = phase2_slice_data(df, 0.0, 1.0, "C")
                batch_full_cycles[(cid, cyc)] = full_c_df

                for mode in ["C", "D"]:
                    for scen_name, (s_bound, e_bound) in scen_bounds.items():
                        for length in lengths:
                            max_start = e_bound - length
                            if max_start < s_bound:
                                continue
                            for start_p in np.arange(s_bound, max_start + 0.0001, step):
                                end_p = start_p + length
                                sliced, soc_l, mode_l = phase2_slice_data(df, start_p, end_p, mode)
                                if not sliced.empty and len(sliced) >= 5:
                                    batch_segments.append({
                                        "cell": cid, "cyc": cyc, "df": sliced,
                                        "soc_label": soc_l, "mode_label": mode_l,
                                        "rul": cell.rul[cyc],
                                        "cycle_key": (cid, cyc)
                                    })

            gc.collect()

        # 배치 단위로 pkl 저장
        print(f"[Batch {batch_id}] Saving {len(batch_segments)} segments to {batch_cache_path}...")
        with open(batch_cache_path, "wb") as f:
            pickle.dump({'segments': batch_segments, 'full_cycles': batch_full_cycles}, f,
                        protocol=pickle.HIGHEST_PROTOCOL)
        print(f"[Batch {batch_id}] Saved successfully.")

        # 전체 집계에 추가 후 배치 메모리 해제
        all_raw_segments.extend(batch_segments)
        full_cycles_map.update(batch_full_cycles)

        del batch_segments, batch_full_cycles
        gc.collect()

    print(f"\nAll batches processed. Total segments: {len(all_raw_segments)}")

In [ ]:
# CACHE_PATH = PROJECT_ROOT / "outputs" / "processed" / f"{DATASET_TYPE}_raw_segments_cache.pkl"
# CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

# if CACHE_PATH.exists():
#     print(f"Loading raw segments from cache: {CACHE_PATH}")
#     with open(CACHE_PATH, "rb") as f:
#         all_raw_segments = pickle.load(f)
#     print(f"Successfully loaded {len(all_raw_segments)} segments.")
# else:
#     scen_bounds = {"H": (0.0, 0.3), "M": (0.3, 0.7), "L": (0.7, 1.0), "Random": (0.0, 1.0)}
#     lengths = [0.1, 0.2, 0.3]
#     step = 0.1

#     all_raw_segments = []

#     for cid, cell in all_cells.items():
#         for cyc in tqdm(sorted(list(cell.data.keys())), desc=f"Phase 2.1: Global Slicing Cell {cid}"):
#             df = cell.data[cyc]
#             full_c_df, _, _ = phase2_slice_data(df, 0.0, 1.0, "C")
            
#             for mode in ["C", "D"]:
#                 for scen_name, (s_bound, e_bound) in scen_bounds.items():
#                     for length in lengths:
#                         max_start = e_bound - length
#                         if max_start < s_bound: continue
#                         for start_p in np.arange(s_bound, max_start + 0.0001, step):
#                             end_p = start_p + length
#                             sliced, soc_l, mode_l = phase2_slice_data(df, start_p, end_p, mode)
#                             if not sliced.empty and len(sliced) >= 5:
#                                 all_raw_segments.append({
#                                     "cell": cid, "cyc": cyc, "df": sliced, 
#                                     "soc_label": soc_l, "mode_label": mode_l, 
#                                     "rul": cell.rul[cyc], "full_cycle_df": full_c_df
#                                 })
    
#     print(f"Slicing complete. Saving {len(all_raw_segments)} segments to cache...")
#     with open(CACHE_PATH, "wb") as f:
#         pickle.dump(all_raw_segments, f)
#     print("Cache saved successfully.")

## Phase 3: 40D Adaptive HI Extraction Logic

In [ ]:
def extract_40d_hi(df, mode="C", history_df=None): 
    v_col = [c for c in df.columns if "Voltage" in c][0]
    i_col = "Current (A)" if "Current (A)" in df.columns else [c for c in df.columns if "Current" in c][0]
    t_col = [c for c in df.columns if "Time" in c][0]
    temp_col = [c for c in df.columns if "Temperature" in c]
    v, i, t = df[v_col].values, df[i_col].values, df[t_col].values
    temp = df[temp_col[0]].values if temp_col else np.full_like(v, 25.0)
    dt = np.diff(t, prepend=t[0])
    his = {}
    
    his["c1_mean_v"], his["c1_var_v"] = np.mean(v), np.var(v)
    his["c1_skew_v"], his["c1_kurt_v"] = skew(v), kurtosis(v)
    his["c1_mean_t"], his["c1_delta_t"] = np.mean(temp), np.max(temp)-np.min(temp)
    his["c1_dtdt"] = np.mean(np.diff(temp)/np.where(dt[1:]>0, dt[1:], 1e-6))
    his["c1_relax_v"] = v[-1] - v[0]
    his["c1_relax_tc"] = np.log(np.abs(v[-1]-v[-2])+1e-6)
    his["c1_var_i"] = np.var(i)
    his["c1_ent_v"] = -np.sum(np.histogram(v, bins=10, density=True)[0] * np.log(np.histogram(v, bins=10, density=True)[0] + 1e-6))
    his["c1_ent_t"] = -np.sum(np.histogram(temp, bins=10, density=True)[0] * np.log(np.histogram(temp, bins=10, density=True)[0] + 1e-6))
    his["c1_de_dq"] = np.sum(v*i*dt) / (np.sum(i*dt)+1e-6)
    his["c1_ma"] = history_df["c1_mean_v"].iloc[-5:].mean() if history_df is not None and not history_df.empty else his["c1_mean_v"]

    for j in range(1, 14): his[f"c2_dis_{j}"] = his[f"c3_cha_{j}"] = 0.0
    if mode == "D":
        c_rate = np.abs(i)/1.1
        his["c2_dis_1"] = np.sum(np.diff(np.round(c_rate, 1)) != 0)
        his["c2_dis_2"] = np.mean(np.abs(np.diff(v)/(np.diff(i)+1e-6)))
        his["c2_dis_3"] = np.trapz(v-np.min(v), t)/(t[-1]-t[0])
        his["c2_dis_4"] = t[np.argmin(v>=3.0)] if np.any(v<3.0) else t[-1]
        his["c2_dis_5"] = np.sum((v<=3.2)&(v>=3.1)) * np.mean(dt)
        dvdt = np.diff(v)/np.where(dt[1:]>0, dt[1:], 1e-6)
        his["c2_dis_6"], his["c2_dis_7"] = np.mean(dvdt), np.var(dvdt)
        his["c2_dis_12"] = np.min(np.diff(v)/np.where(np.diff(np.cumsum(np.abs(i)*dt)/3600.0)>0, np.diff(np.cumsum(np.abs(i)*dt)/3600.0), 1e-6))
    elif mode == "C":
        is_cv, q_cum = np.abs(np.diff(v))<0.001, np.cumsum(np.abs(i)*dt)/3600.0
        if np.any(is_cv):
            cv_s = np.where(is_cv)[0][0]
            his["c3_cha_1"], his["c3_cha_2"], his["c3_cha_3"] = q_cum[cv_s]/q_cum[-1], t[cv_s]-t[0], t[-1]-t[cv_s]
        his["c3_cha_8"] = q_cum[(v>=3.35)&(v<=3.45)][-1] - q_cum[(v>=3.35)&(v<=3.45)][0] if np.any((v>=3.35)&(v<=3.45)) else 0.0
        dqdv = np.diff(q_cum)/np.where(np.diff(v)>0.0001, np.diff(v), 1e-6)
        his["c3_cha_9"], his["c3_cha_10"] = np.max(dqdv), np.min(dqdv)
    return pd.Series(his)

## Phase 4: Feature Extraction & Scaling (Reusing Sliced Segments)

This phase iterates through the pool from Phase 2.1. No redundant slicing is performed here.

In [ ]:
SCALING_METHOD = "zscore"  # Choose 'zscore' or 'minmax'

HI_CACHE_DIR = PROJECT_ROOT / "outputs" / "processed" / f"{DATASET_TYPE}_chunks"
HI_FINAL_CACHE_PATH = PROJECT_ROOT / "outputs" / "processed" / f"{DATASET_TYPE}_feature_pool_cache.pkl"

if HI_FINAL_CACHE_PATH.exists():
    print(f"Loading feature pool from cache: {HI_FINAL_CACHE_PATH}")
    with open(HI_FINAL_CACHE_PATH, "rb") as f:
        hi_cache = pickle.load(f)
    feature_pool = hi_cache['feature_pool']
    scaler = hi_cache['scaler']
    print(f"Successfully loaded {len(feature_pool)} features.")

else:
    batch_files = sorted(HI_CACHE_DIR.glob("batch_*.pkl"))
    assert batch_files, f"No batch files found in {HI_CACHE_DIR}. Run slicing step first."

    # cell_histories는 셀 단위로 누적되어야 하므로 전역으로 유지
    cell_histories = {cid: pd.DataFrame() for cid in all_cells.keys()}
    feature_pool = []

    # --- Phase 4-1: 배치별 HI 추출 ---
    for batch_file in sorted(batch_files):
        batch_id = batch_file.stem  # e.g. "batch_1"
        print(f"\n[{batch_id}] Extracting HI features...")

        with open(batch_file, "rb") as f:
            batch_data = pickle.load(f)

        batch_segments = batch_data['segments']

        for item in tqdm(batch_segments, desc=f"  Phase 4: HI Extraction [{batch_id}]"):
            cid = item['cell']
            mode_str = "C" if item['mode_label'] == 1 else "D"
            hi = extract_40d_hi(item['df'], mode_str, cell_histories[cid])

            feature_pool.append({
                "cell": cid, "cyc": item['cyc'], "raw_x": hi,
                "soc_label": item['soc_label'], "mode_label": item['mode_label'],
                "y": item['rul']
            })

        del batch_data, batch_segments
        gc.collect()

    # --- Phase 4-2: 전체 기준으로 스케일러 fit & transform ---
    print(f"\nTotal features extracted: {len(feature_pool)}")
    print(f"Fitting scaler ({SCALING_METHOD}) on full feature pool...")

    raw_x_df = pd.DataFrame([x['raw_x'] for x in feature_pool])
    scaler = StandardScaler() if SCALING_METHOD == "zscore" else MinMaxScaler()
    scaled_x = scaler.fit_transform(raw_x_df)

    for i, item in enumerate(feature_pool):
        item['scaled_x'] = scaled_x[i]

    del raw_x_df, scaled_x
    gc.collect()

    # --- Phase 4-3: 캐시 저장 ---
    print(f"Saving feature pool to {HI_FINAL_CACHE_PATH}...")
    with open(HI_FINAL_CACHE_PATH, "wb") as f:
        pickle.dump({'feature_pool': feature_pool, 'scaler': scaler}, f,
                    protocol=pickle.HIGHEST_PROTOCOL)
    print("Feature pool cache saved successfully.")

print(f"Extraction and Scaling ({SCALING_METHOD}) complete. Total: {len(feature_pool)} items.")

## Phase 5: Final Saving

In [ ]:
def save_processed_dataset(pool, dataset_type):
    out_path = PROJECT_ROOT / "outputs" / "processed" / f"{dataset_type}_optimized_tensors.pkl"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    final_data = []
    for item in pool:
        final_data.append({
            "cell": item['cell'], "cyc": item['cyc'], 
            "x": item['scaled_x'], "soc_label": item['soc_label'], 
            "mode_label": item['mode_label'], "y": item['y']
        })
    with open(out_path, "wb") as f: pickle.dump(final_data, f)
    print(f"Saved to {out_path}")

save_processed_dataset(feature_pool, DATASET_TYPE)